# Determining appropriate metric thresholds

The purpose of this notebook is to derive the minimum threshold parameter for the metrics. The logic was adapted per DeepEval documentation for the answer_correctness metric.

Steps:
1. Re-ran data_generation/main.py to generate answers alongisde retrieved chunks.
2. Manually labelled the dataset for the answer_relevancy, context_relevancy, and faithfulness metrics. The labelling for cultural_neutrality and tone_attunement were retrieved through surveys sent out to RC members where they were asked to annotate the QA pairs.
3. Used that labeling to derive an expected pass rate — e.g. if 75% of manually reviewed cases are acceptable, target is a 75th percentile threshold.
4. Ran main.py to generate the evaluation framework metric scores.
5. Compared results.
6. Discussion and next steps.

# Extracting metric scores from DeepEval

The scores will be calculated in two parts: the humanitarian metrics on Dataset A (`qa_pairs.csv`), because they were annotated by RC members; answer_relevancy, context_relevancy, and faithfulness on Dataset B (`qa_pairs_annotated.csv`), because they contain the context for the answers, and they were manually annotated by me.

Here the scores are 

In [163]:
import os
import ast
import pandas as pd
from rich import print
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import AzureOpenAIModel
from deepeval.metrics import AnswerRelevancyMetric, ContextualRelevancyMetric, FaithfulnessMetric, GEval

In [241]:
# Loading Dataset B
data = pd.read_csv('../data/qa_pairs_annotated.csv')

In [60]:
pd.set_option('display.max_colwidth', None) # Show full content of each cell

In [4]:
# CUSTOM MODEL
endpoint = "https://510-ai-research.openai.azure.com/"
model = "gpt-4.1"
deployment = "gpt-4.1-students"
# loading variables from .env file
subscription_key = os.getenv("azure_subscription_key")
api_version = "2024-12-01-preview"
# model
custom_model = AzureOpenAIModel(
  model=deployment,
  api_key=subscription_key,
  azure_endpoint=endpoint,
  api_version=api_version,
  deployment_name=deployment
)

In [5]:
# TEST CASE CREATION
test_cases = []
for i in range(len(data)):
    tc = LLMTestCase(
        input=data['user_input'][i],
        actual_output=data['bot_output'][i],
        retrieval_context=ast.literal_eval(data['context'][i]),
    )
    test_cases.append(tc)

## Metric Calculation

Default threshold from DeepEval is 0.5.

In [6]:
ar = AnswerRelevancyMetric(
    threshold=0.5,
    include_reason=True,
    model=custom_model
)
cr = ContextualRelevancyMetric(
    threshold=0.5,
    include_reason=True,
    model=custom_model
)
f = FaithfulnessMetric(
    threshold=0.5,
    include_reason=True,
    model=custom_model
)

evaluation_results = evaluate(test_cases=test_cases, metrics=[ar, cr, f])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 0.75, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 0.75 because while the response partially addressed the user's request for a doctor, it included irrelevant information about the chatbot's name, which did not contribute to solving the user's need., error: None)
  - ❌ Contextual Relevancy (score: 0.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 0.00 because there are no relevant statements in the retrieval context and no reasons provided for irrelevancy., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 1.00 because there are no contradictions—the actual output aligns perfectly with the retrieval context. Great job!, error: None)

For test case:

  - input: I need a doctor.
  - actual output: Hello! I'm Aidly, the HIA chatbot. I can help you fi

⚠ WARNING: No hyperparameters logged.
» ]8;id=354486;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 20.66s | token cost: None)
» Test Results (21 total tests):
   » Pass Rate: 19.05% | Passed: 4 | Failed: 17

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [7]:
results = []
for test_result in evaluation_results.test_results:
    row = {
        "input": test_result.input,
        "bot_output": test_result.actual_output,
        "retrieval_context": test_result.retrieval_context
    }
    for metric_data in test_result.metrics_data:
        row[metric_data.name] = metric_data.score
        row[metric_data.name + "_reason"] = metric_data.reason
    results.append(row)
results_df = pd.DataFrame(results)

In [8]:
results_df.head()

,input,bot_output,retrieval_context,Answer Relevancy,Answer Relevancy_reason,Contextual Relevancy,Contextual Relevancy_reason,Faithfulness,Faithfulness_reason
0,I need a doctor.,"Hello! I'm Aidly, the HIA chatbot. I can help ...",[],0.75,The score is 0.75 because while the response p...,0.0,The score is 0.00 because there are no relevan...,1.0,The score is 1.00 because there are no contrad...
1,I am pregnant and my landlord kicked me out be...,I'm really sorry to hear about your situation....,[],1.00,The score is 1.00 because the response was ful...,0.0,The score is 0.00 because there are no relevan...,1.0,The score is 1.00 because there are no contrad...
2,I don't have a place to stay because I escaped...,I'm really sorry to hear about your situation....,[],1.00,The score is 1.00 because every part of the re...,0.0,The score is 0.00 because there are no relevan...,1.0,The score is 1.00 because there are no contrad...
3,I need food!,"Hi! I'm Aidly, the HIA chatbot. How can I assi...",[],0.80,"The score is 0.80 because, while the response ...",0.0,The score is 0.00 because there are no relevan...,1.0,The score is 1.00 because there are no contrad...
4,You referred me to an organisation for asylum ...,I'm sorry to hear about your difficult situati...,[],1.00,Great job! The score is 1.00 because every par...,0.0,The score is 0.00 because there are no relevan...,1.0,The score is 1.00 because there are no contrad...


In [9]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   input                        21 non-null     object 
 1   bot_output                   21 non-null     object 
 2   retrieval_context            21 non-null     object 
 3   Answer Relevancy             21 non-null     float64
 4   Answer Relevancy_reason      21 non-null     object 
 5   Contextual Relevancy         21 non-null     float64
 6   Contextual Relevancy_reason  21 non-null     object 
 7   Faithfulness                 21 non-null     float64
 8   Faithfulness_reason          21 non-null     object 
dtypes: float64(3), object(6)
memory usage: 1.6+ KB


## Merging the two together

In [10]:
data = data.rename(columns={"user_input": "input"})
merged = results_df.merge(data, on="input", how="inner")

In [11]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   input                        21 non-null     object 
 1   bot_output_x                 21 non-null     object 
 2   retrieval_context            21 non-null     object 
 3   Answer Relevancy             21 non-null     float64
 4   Answer Relevancy_reason      21 non-null     object 
 5   Contextual Relevancy         21 non-null     float64
 6   Contextual Relevancy_reason  21 non-null     object 
 7   Faithfulness                 21 non-null     float64
 8   Faithfulness_reason          21 non-null     object 
 9   bot_output_y                 21 non-null     object 
 10  context                      21 non-null     object 
 11  answer_relevancy_label       21 non-null     int64  
 12  context_relevancy_label      13 non-null     float64
 13  faithfulness_label    

In [12]:
merged.drop(columns=["bot_output_y", "context"], inplace=True)
merged.rename(columns={"bot_output_x": "bot_output"}, inplace=True)

In [13]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   input                        21 non-null     object 
 1   bot_output                   21 non-null     object 
 2   retrieval_context            21 non-null     object 
 3   Answer Relevancy             21 non-null     float64
 4   Answer Relevancy_reason      21 non-null     object 
 5   Contextual Relevancy         21 non-null     float64
 6   Contextual Relevancy_reason  21 non-null     object 
 7   Faithfulness                 21 non-null     float64
 8   Faithfulness_reason          21 non-null     object 
 9   answer_relevancy_label       21 non-null     int64  
 10  context_relevancy_label      13 non-null     float64
 11  faithfulness_label           13 non-null     float64
dtypes: float64(5), int64(1), object(6)
memory usage: 2.1+ KB


## Answer Relevancy

In [14]:
# Extract the labelling column
label = merged["answer_relevancy_label"]

# Calculate the percentage of entries with answer_relevancy_label = 1
count = 0
for val in label:
    if val == 1:
        count += 1
print(f"Number of entries with answer_relevancy_label = 1: {count}")
total_entries = len(label)
print(f"Total number of entries: {total_entries}")
relevancy_percentile = (count / total_entries) * 100

print(f"Percentage of entries with answer_relevancy_label = 1 from manual annotation: {relevancy_percentile:.2f}%")

Number of entries with answer_relevancy_label = 1: 20

Total number of entries: 21

Percentage of entries with answer_relevancy_label = 1 from manual annotation: 95.24%

In [15]:
cols_to_display = ["input", "bot_output", "Answer Relevancy", "Answer Relevancy_reason", "answer_relevancy_label"]
ar = merged[cols_to_display]

In [16]:
threshold = 0.5  # DeepEval default

# Pass rate (from labels)
my_pass_rate = (ar["answer_relevancy_label"] == 1).mean()

# DeepEval's pass rate (at default threshold)
deepeval_pass_rate = (ar["Answer Relevancy"] >= threshold).mean()

print(f"My pass rate: {my_pass_rate:.2%}")
print(f"DeepEval pass rate at threshold {threshold}: {deepeval_pass_rate:.2%}")

My pass rate: 95.24%

DeepEval pass rate at threshold 0.5: 95.24%

### Do we agree on which instances pass and which fail?

Passing labels

In [ ]:
print(f"Passing rows (my labels): {ar[ar['answer_relevancy_label'] == 1].index.tolist()}")
print(f"Passing rows (Deep Eval): {ar[ar['Answer Relevancy'] >= threshold].index.tolist()}")
print(f"Rows which don't match: {set(ar[ar['answer_relevancy_label'] == 1].index) ^ set(ar[ar['Answer Relevancy'] >= threshold].index)}")

Passing rows (my labels): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20]

Passing rows (Deep Eval): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20]

Rows which don't don't match: set()

Non-passing labels

In [28]:
print(f"Non-passing rows (my labels): {ar[ar['answer_relevancy_label'] == 0].index.tolist()}")
print(f"Non-passing rows (Deep Eval): {ar[ar['Answer Relevancy'] < threshold].index.tolist()}")
print(f"Rows which don't don't match: {set(ar[ar['answer_relevancy_label'] == 0].index) ^ set(ar[ar['Answer Relevancy'] < threshold].index)}")

Non-passing rows (my labels): [13]

Non-passing rows (Deep Eval): [13]

Rows which don't don't match: set()

Non-passing labels

The passes and failures overalap perfectly, therefore we can compute the target threshold directly.

### Applying the percentile method with target

In [239]:
scores = ar["Answer Relevancy"].tolist()
sorted_scores = sorted(scores)
print(sorted_scores)

[
    0.0,
    0.7333333333333333,
    0.75,
    0.8,
    0.8,
    0.8181818181818182,
    0.8333333333333334,
    0.8571428571428571,
    0.9166666666666666,
    0.9166666666666666,
    0.9230769230769231,
    0.9375,
    0.9411764705882353,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0
]

Sorted ascending, `scores[1]` is the second-lowest score = 0.73.

DeepEval's formula:

In [150]:
index = int(len(sorted_scores) * (1 - relevancy_percentile / 100))

In [151]:
index

1

In [240]:
print(f"Answer Relevance threshold for {relevancy_percentile:.2f}% percentile: {sorted_scores[index]:.2f}")

Answer Relevance threshold for 95.24% percentile: 0.86

## Contextual Relevancy

In [111]:
# Extract the labelling column
label = merged["context_relevancy_label"]

# Calculate the percentage of entries with context_relevancy_label = 1
count = 0
for val in label:
    if val == 1:
        count += 1
print(f"Number of entries with context_relevancy_label = 1: {count}")

def has_context(ctx):
  if isinstance(ctx, str):
    ctx = ctx.strip()
    if ctx in ("", "[]"):
      return False
    try:
      ctx = ast.literal_eval(ctx)
    except Exception:
      return True
  return bool(ctx)

total_entries = merged["retrieval_context"].apply(has_context).sum()
print(f"Total number of entries (answers with context chunks): {total_entries}")
context_relevancy_percentile = (count / total_entries) * 100

print(f"Percentage of entries with context_relevancy_label = 1 from manual annotation: {context_relevancy_percentile:.2f}%")

Number of entries with context_relevancy_label = 1: 2

Total number of entries (answers with context chunks): 13

Percentage of entries with context_relevancy_label = 1 from manual annotation: 15.38%

In [112]:
cols_to_display = ["input", "bot_output", "retrieval_context", "Contextual Relevancy", "Contextual Relevancy_reason", "context_relevancy_label"]
cr = merged[cols_to_display]

In [113]:
threshold = 0.5  # DeepEval default
cr_labelled = cr[cr["context_relevancy_label"].notna()] # to not catch NaN-labelled rows as mismatches

# Pass rate (from labels)
my_pass_rate = (cr_labelled["context_relevancy_label"] == 1).mean()

# DeepEval's pass rate (at default threshold)
deepeval_pass_rate = (cr_labelled["Contextual Relevancy"] >= threshold).mean()

print(f"My pass rate: {my_pass_rate:.2%}")
print(f"DeepEval pass rate at threshold {threshold}: {deepeval_pass_rate:.2%}")

My pass rate: 15.38%

DeepEval pass rate at threshold 0.5: 30.77%

### Do we agree on which instances pass and which fail?

Passing labels

In [93]:
labelled = cr[cr['context_relevancy_label'].notna()] # to not catch NaN-labelled rows as mismatches
print(f"Passing rows (my labels): {labelled[labelled['context_relevancy_label'] == 1].index.tolist()}")
print(f"Passing rows (Deep Eval): {labelled[labelled['Contextual Relevancy'] >= threshold].index.tolist()}")
print(f"Rows which don't don't match: {labelled[(labelled['context_relevancy_label'] == 1) !=(labelled['Contextual Relevancy'] >= threshold)].index.tolist()}")

Passing rows (my labels): [8, 12]

Passing rows (Deep Eval): [8, 11, 12, 16]

Rows which don't don't match: [11, 16]

In [99]:
non_matches = labelled[(labelled['context_relevancy_label'] == 1) != (labelled['Contextual Relevancy'] >= threshold)].index
print(f"Non-matching rows (Rows I matched as non-contextually relevant): {cr.loc[non_matches, ['input', 'retrieval_context', 'Contextual Relevancy', 'Contextual Relevancy_reason' ]]}")

Non-matching rows (Rows I matched as non-contextually relevant):                                                   
input  \
11  I am from Ukraine and I work in a factory. I live with my other colleagues, and my boss pays for my rent, 
utilities, and gives me an allowance. I want to go home this Christmas to visit my family, but he refuses because 
he says it's the busiest season and if I leave everyone will want to go home as well, so we will all lose our jobs.
Can you help somehow?   
16                                                                                                                 
I have been to 3 hospitals, but they refuse me because I am undocumented. I am in immense pain and need immediate 
attention. I am in Amsterdam. What do I do?   

                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                 

After assesing the LLM reasoning, I decided to agree and flip row 11 from fail to pass. Therefore, we must recalculate. However, for row 16, DeepEval was too lenient, as a big portion of the retrieved contexts (such as information about sexual health) is irrelevant to a user who has already been redirected by the bot regarding medical procedures.

In [118]:
merged.loc[11, 'context_relevancy_label'] = 1 # general change
cr_labelled.loc[11, 'context_relevancy_label'] = 1 # to reflect the change in the labelled dataframe

#  Pass rate (from labels)
my_pass_rate = (cr_labelled["context_relevancy_label"] == 1).mean()

# DeepEval's pass rate (at default threshold)
deepeval_pass_rate = (cr_labelled["Contextual Relevancy"] >= threshold).mean()

print(f"My pass rate: {my_pass_rate:.2%}")
print(f"DeepEval pass rate at threshold {threshold}: {deepeval_pass_rate:.2%}")

My pass rate: 23.08%

DeepEval pass rate at threshold 0.5: 30.77%

### Applying the percentile method with target

In [145]:
scores = cr_labelled["Contextual Relevancy"].tolist()
sorted_scores = sorted(scores)
print(sorted_scores)

[
    0.016,
    0.16129032258064516,
    0.24271844660194175,
    0.31443298969072164,
    0.391304347826087,
    0.39344262295081966,
    0.3968253968253968,
    0.4146341463414634,
    0.48,
    0.5483870967741935,
    0.797979797979798,
    0.8691588785046729,
    0.8699186991869918
]

In [146]:
index = int(len(sorted_scores) * (1 - context_relevancy_percentile / 100))

In [147]:
index

11

This gives the 11th-lowest score (out of 13) as the threshold. Only scores above the 10th-lowest pass.

In [148]:
print(f"Threshold for {context_relevancy_percentile:.2f}% percentile: {sorted_scores[index]:.2f}")

Threshold for 15.38% percentile: 0.87

In [124]:
# Sanity check
cr_labelled["de_predicted"] = cr_labelled["Contextual Relevancy"] >= 0.87
print(cr_labelled[["input", "Contextual Relevancy", "context_relevancy_label", "de_predicted"]])

C:\Users\dari\AppData\Local\Temp\ipykernel_39000\946920461.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cr_labelled["de_predicted"] = cr_labelled["Contextual Relevancy"] >= 0.87


input  \
8                                                                                                                  
I'm homeless and looking for a shelter. I am very tired. I don't have anyone to help me.   
9                                                                                                                  
You already told me about the day shelters. I know about those. I need an actual place to sleep tonight with my 
children. Stop giving me the same information!   
10                                                                                                                 
live in Utrecht, but I got kicked out of my house today. Where can I find a warm meal?   
11  I am from Ukraine and I work in a factory. I live with my other colleagues, and my boss pays for my rent, 
utilities, and gives me an allowance. I want to go home this Christmas to visit my family, but he refuses because 
he says it's the busiest season and if I leave everyone will want to go home as well, so we will all lose our jobs.
Can you help somehow?   
12                                                                                                                 
My family and I got denied access to the shelter. We are from Ukraine and we are in Rotterdam right now. We are 
very tired.   
13                                                                                                                 
wer find winter clodes. I nigeria.   
14                                                                                                                 
I am from Morocco and I have been living in the Netherlands for 20 years. My doctor retired and the new practice is
refusing to register me because I don't have a BSN. I know my rights, and this is incredibly frustrating. What else
can I do?   
15                                                                                                                 
I have diabetes and I am undocumented. I need insulin but I also cannot afford the food I need for my diet. The 
doctor told me I need to eat differently but everything is too expensive.   
16                                                                                                                 
I have been to 3 hospitals, but they refuse me because I am undocumented. I am in immense pain and need immediate 
attention. I am in Amsterdam. What do I do?   
17                                                                                                                 
I have a temporary residence permit from Portugal, but I am originally from Iraq. I moved to the Netherlands to 
find work, but now they are telling me I cannot work here. I don’t understand. I have European papers.   
18                                                                           I live in Utrecht and I am on the 
streets now because of an abusive landlord. I went to a homeless shelter with my children and they refused me 
because I still have my job and some saved up money, but not enough for a hotel. Now they are threatening to take 
my children away. Help me!   
19                                                                                                                 
I am undocumented and I have two children. They need to go to school but I am afraid to register them. Also, where 
can I get winter clothes for them? They are growing fast   
20                                                                                                                 
I lost my job and got kicked out of my house because I could not pay rent. I am from Poland. I have been homeless 
for a couple of days, and need to shower and do laundry.   

    Contextual Relevancy  context_relevancy_label  de_predicted  
8               0.869919                      1.0         False  
9               0.480000                      0.0         False  
10              0.161290                      0.0         False  
11              0.869159             

Most of rows that failed sit just below 0.87, therefore we adjust the threshold to 0.86.

In [125]:
cr_labelled["de_predicted"] = cr_labelled["Contextual Relevancy"] >= 0.86
print(cr_labelled[["input", "Contextual Relevancy", "context_relevancy_label", "de_predicted"]])

C:\Users\dari\AppData\Local\Temp\ipykernel_39000\4215209069.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cr_labelled["de_predicted"] = cr_labelled["Contextual Relevancy"] >= 0.86


input  \
8                                                                                                                  
I'm homeless and looking for a shelter. I am very tired. I don't have anyone to help me.   
9                                                                                                                  
You already told me about the day shelters. I know about those. I need an actual place to sleep tonight with my 
children. Stop giving me the same information!   
10                                                                                                                 
live in Utrecht, but I got kicked out of my house today. Where can I find a warm meal?   
11  I am from Ukraine and I work in a factory. I live with my other colleagues, and my boss pays for my rent, 
utilities, and gives me an allowance. I want to go home this Christmas to visit my family, but he refuses because 
he says it's the busiest season and if I leave everyone will want to go home as well, so we will all lose our jobs.
Can you help somehow?   
12                                                                                                                 
My family and I got denied access to the shelter. We are from Ukraine and we are in Rotterdam right now. We are 
very tired.   
13                                                                                                                 
wer find winter clodes. I nigeria.   
14                                                                                                                 
I am from Morocco and I have been living in the Netherlands for 20 years. My doctor retired and the new practice is
refusing to register me because I don't have a BSN. I know my rights, and this is incredibly frustrating. What else
can I do?   
15                                                                                                                 
I have diabetes and I am undocumented. I need insulin but I also cannot afford the food I need for my diet. The 
doctor told me I need to eat differently but everything is too expensive.   
16                                                                                                                 
I have been to 3 hospitals, but they refuse me because I am undocumented. I am in immense pain and need immediate 
attention. I am in Amsterdam. What do I do?   
17                                                                                                                 
I have a temporary residence permit from Portugal, but I am originally from Iraq. I moved to the Netherlands to 
find work, but now they are telling me I cannot work here. I don’t understand. I have European papers.   
18                                                                           I live in Utrecht and I am on the 
streets now because of an abusive landlord. I went to a homeless shelter with my children and they refused me 
because I still have my job and some saved up money, but not enough for a hotel. Now they are threatening to take 
my children away. Help me!   
19                                                                                                                 
I am undocumented and I have two children. They need to go to school but I am afraid to register them. Also, where 
can I get winter clothes for them? They are growing fast   
20                                                                                                                 
I lost my job and got kicked out of my house because I could not pay rent. I am from Poland. I have been homeless 
for a couple of days, and need to shower and do laundry.   

    Contextual Relevancy  context_relevancy_label  de_predicted  
8               0.869919                      1.0          True  
9               0.480000                      0.0         False  
10              0.161290                      0.0         False  
11              0.869159             

Final threshold: 0.86.

## Faithfulness

In [126]:
# Extract the labelling column
label = merged["faithfulness_label"]

# Calculate the percentage of entries with faithfulness_label = 1
count = 0
for val in label:
    if val == 1:
        count += 1
total_entries = merged["retrieval_context"].apply(has_context).sum()
print(f"Total number of entries (answers with context chunks): {total_entries}")
faithfulness_percentile = (count / total_entries) * 100

print(f"Percentage of entries with faithfulness_label = 1 from manual annotation: {faithfulness_percentile:.2f}%")

Total number of entries (answers with context chunks): 13

Percentage of entries with faithfulness_label = 1 from manual annotation: 92.31%

In [79]:
cols_to_display = ["input", "bot_output", "retrieval_context", "Faithfulness", "Faithfulness_reason", "faithfulness_label"]
faith = merged[cols_to_display]

In [155]:
threshold = 0.5  # DeepEval default
faith_labelled = faith[faith["faithfulness_label"].notna()] # to not catch NaN-labelled rows as mismatches

# Pass rate (from labels)
my_pass_rate = (faith_labelled["faithfulness_label"] == 1).mean()

# DeepEval's pass rate (at default threshold)
deepeval_pass_rate = (faith_labelled["Faithfulness"] >= threshold).mean()

print(f"My pass rate: {my_pass_rate:.2%}")
print(f"DeepEval pass rate at threshold {threshold}: {deepeval_pass_rate:.2%}")

My pass rate: 92.31%

DeepEval pass rate at threshold 0.5: 100.00%

### Do we agree on which instances pass and which fail?

Passing labels

In [84]:
labelled = faith[faith['faithfulness_label'].notna()] # to not catch NaN-labelled rows as mismatches
print(f"Passing rows (my labels): {labelled[labelled['faithfulness_label'] == 1].index.tolist()}")
print(f"Passing rows (Deep Eval): {labelled[labelled['Faithfulness'] >= threshold].index.tolist()}")
print(f"Rows which don't don't match: {labelled[(labelled['faithfulness_label'] == 1) !=(labelled['Faithfulness'] >= threshold)].index.tolist()}")

Passing rows (my labels): [8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Passing rows (Deep Eval): [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Rows which don't don't match: [11]

In [83]:
non_matches = labelled[(labelled['faithfulness_label'] == 1) != (labelled['Faithfulness'] >= threshold)].index
print(f"Non-matching rows (Rows I matched as non-faithful): {faith.loc[non_matches, ['input','bot_output', 'retrieval_context', 'Faithfulness', 'Faithfulness_reason' ]]}")

Non-matching rows (Rows I matched as non-faithful):                                                                
input  \
11  I am from Ukraine and I work in a factory. I live with my other colleagues, and my boss pays for my rent, 
utilities, and gives me an allowance. I want to go home this Christmas to visit my family, but he refuses because 
he says it's the busiest season and if I leave everyone will want to go home as well, so we will all lose our jobs.
Can you help somehow?   

                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                           bot_output  \
11  Hello! I am Aidly, the HIA chatbot.\n\nFrom the information available, as a displaced person from Ukraine 
working in the Netherlands, you have the same labour rights as Dutch employees, including the right to a safe 
working space and fair treatment. Your employer should not force you to work against your will or prevent you from 
taking reasonable time off, such as for family visits during holidays.\n\nIf your employer is refusing to allow you
to visit your family for Christmas and you feel pressured or unsafe, you can seek help. There are organizations 
like FairWork that specialize in helping people with work problems or labour exploitation. You can contact FairWork
at +31 20 760 08 09 or visit https://www.fairwork.nu/en/homepage/ for assistance.\n\nAlso, you can share with your 
employer the flyer "Know your rights when you work," available in English and Ukrainian, to inform them about your 
rights: https://www.comensha.nl/media/jizlbk4d/2026-know-your-rights-flyer-oek-ontheemden-eng.pdf\n\nIf the 
situation is urgent or you feel forced to work against your will, you can ask for immediate help through 
FairWork.\n\nRemember, you should also report your departure if you live in a shelter, but since you live with 
colleagues and your boss pays rent, make sure to inform the relevant reception or municipality if applicable.\n\nIf
you want, I can help you find more detailed advice or contacts. Would you like that?   

                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                               

12 rows were labelled pass and 1 fail. The single fail is a response asserting a right to time off for family visits — true under Dutch labour law but not present in the retrieved chunks, which list only safe working space, regular breaks, and minimum wage.

With a label-derived target pass rate of 92.3%, the percentile formula returns a threshold of 1.00. Applied back to the data, this threshold passes all 13 rows including this case. The percentile method cannot discriminate on a constant distribution.

### Using synthetic data to determine Faithfulness threshold

We therefore take a different methodological approach, where we use the evaluated synthetic dataset to set the metric threshold.

In [128]:
synthetic = pd.read_csv('../data/goldens_final.csv')
synthetic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   user_input       101 non-null    object
 1   expected_output  101 non-null    object
 2   bot_output       101 non-null    object
 3   context          101 non-null    object
dtypes: object(4)
memory usage: 3.3+ KB


In [129]:
# TEST CASE CREATION
test_cases = []
for i in range(len(synthetic)):
    tc = LLMTestCase(
        input=synthetic['user_input'][i],
        actual_output=synthetic['bot_output'][i],
        retrieval_context=ast.literal_eval(synthetic['context'][i]),
    )
    test_cases.append(tc)

In [131]:
evaluation_results = evaluate(test_cases=test_cases, metrics=[f])

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 1.00 because there are no contradictions—the actual output aligns perfectly with the retrieval context. Great job!, error: None)

For test case:

  - input: How do limited rights, lack of benefits, and housing restrictions increase UM vulnerability to exploitation?
  - actual output: I don't have the right information to answer your question.
  - expected output: None
  - context: None
  - retrieval context: ["Document: I have a problem at work and/or I don't feel safe while working or there is labour exploitation.\n\nWhat should I do?\n\nIt’s very important for you to know your rights and feel secure in any situation.\n\nDo you need help or are you unsure about your work?\n\nAre you worried about someone else?    \n\nOpora (https://www.oporafoundation.nl/) provides juridical information to displaced persons from Ukraine on topics related to

⚠ WARNING: No hyperparameters logged.
» ]8;id=750785;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 94.99s | token cost: None)
» Test Results (101 total tests):
   » Pass Rate: 96.04% | Passed: 97 | Failed: 4

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [132]:
results = []
for test_result in evaluation_results.test_results:
    row = {
        "input": test_result.input,
        "bot_output": test_result.actual_output,
        "retrieval_context": test_result.retrieval_context
    }
    for metric_data in test_result.metrics_data:
        row[metric_data.name] = metric_data.score
        row[metric_data.name + "_reason"] = metric_data.reason
    results.append(row)
synthetic_results_df = pd.DataFrame(results)

In [133]:
synthetic_results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   input                101 non-null    object 
 1   bot_output           101 non-null    object 
 2   retrieval_context    101 non-null    object 
 3   Faithfulness         101 non-null    float64
 4   Faithfulness_reason  101 non-null    object 
dtypes: float64(1), object(4)
memory usage: 4.1+ KB


In [134]:
print(f"n = {len(synthetic_results_df['Faithfulness'])}")
print(synthetic_results_df['Faithfulness'].describe())
print(f"Count at 1.0: {(synthetic_results_df['Faithfulness'] == 1.0).sum()}")

n = 101

count    101.000000
mean       0.949046
std        0.154514
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: Faithfulness, dtype: float64

Count at 1.0: 81

The distribution is at a state where we can compute a non-degenerate threshold.

In [236]:
sorted_scores = sorted(synthetic_results_df['Faithfulness'].tolist())  # ascending
index = int(len(sorted_scores) * (1 - faithfulness_percentile / 100))

In [237]:
index

7

The threshold is the 8th-lowest score.

In [238]:
print(f"Faithfulness threshold for {faithfulness_percentile:.2f}% percentile: {sorted_scores[index]:.2f}")

Faithfulness threshold for 92.31% percentile: 0.85

In [156]:
# Sanity check
faith_labelled["de_predicted"] = faith_labelled["Faithfulness"] >= 0.85
print(faith_labelled[["input", "Faithfulness", "faithfulness_label", "de_predicted"]])

C:\Users\dari\AppData\Local\Temp\ipykernel_39000\378777663.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  faith_labelled["de_predicted"] = faith_labelled["Faithfulness"] >= 0.85


input  \
8                                                                                                                  
I'm homeless and looking for a shelter. I am very tired. I don't have anyone to help me.   
9                                                                                                                  
You already told me about the day shelters. I know about those. I need an actual place to sleep tonight with my 
children. Stop giving me the same information!   
10                                                                                                                 
live in Utrecht, but I got kicked out of my house today. Where can I find a warm meal?   
11  I am from Ukraine and I work in a factory. I live with my other colleagues, and my boss pays for my rent, 
utilities, and gives me an allowance. I want to go home this Christmas to visit my family, but he refuses because 
he says it's the busiest season and if I leave everyone will want to go home as well, so we will all lose our jobs.
Can you help somehow?   
12                                                                                                                 
My family and I got denied access to the shelter. We are from Ukraine and we are in Rotterdam right now. We are 
very tired.   
13                                                                                                                 
wer find winter clodes. I nigeria.   
14                                                                                                                 
I am from Morocco and I have been living in the Netherlands for 20 years. My doctor retired and the new practice is
refusing to register me because I don't have a BSN. I know my rights, and this is incredibly frustrating. What else
can I do?   
15                                                                                                                 
I have diabetes and I am undocumented. I need insulin but I also cannot afford the food I need for my diet. The 
doctor told me I need to eat differently but everything is too expensive.   
16                                                                                                                 
I have been to 3 hospitals, but they refuse me because I am undocumented. I am in immense pain and need immediate 
attention. I am in Amsterdam. What do I do?   
17                                                                                                                 
I have a temporary residence permit from Portugal, but I am originally from Iraq. I moved to the Netherlands to 
find work, but now they are telling me I cannot work here. I don’t understand. I have European papers.   
18                                                                           I live in Utrecht and I am on the 
streets now because of an abusive landlord. I went to a homeless shelter with my children and they refused me 
because I still have my job and some saved up money, but not enough for a hotel. Now they are threatening to take 
my children away. Help me!   
19                                                                                                                 
I am undocumented and I have two children. They need to go to school but I am afraid to register them. Also, where 
can I get winter clothes for them? They are growing fast   
20                                                                                                                 
I lost my job and got kicked out of my house because I could not pay rent. I am from Poland. I have been homeless 
for a couple of days, and need to shower and do laundry.   

    Faithfulness  faithfulness_label  de_predicted  
8            1.0                 1.0          True  
9            1.0                 1.0          True  
10           1.0                 1.0          True  
11           1.0                 0.0          True  
12           1.0                 1.0

# Humanitarian metrics

First, we calculate the metrics on Dataset A.

In [196]:
# TEST CASE CREATION
test_cases = []
for i in range(len(data)):
    tc = LLMTestCase(
        input=data['user_input'][i],
        actual_output=data['bot_output'][i],
    )
    test_cases.append(tc)

In [197]:
ta = GEval(
    name="Tone Attunement",
    criteria="Determine whether the chatbot communication style appropriately matches the emotional register of the user's query. Consider factors such as formality, empathy, and tone consistency in your evaluation.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=custom_model
)

cn = GEval(
    name="Cultural Neutrality",
    criteria="Determine whether responses are free from assumptions about the user's background, legal status, literacy, gender, or cultural context that could introduce bias or alienate vulnerable populations.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=custom_model
)

humanitarian_results = evaluate(test_cases=test_cases, metrics=[ta, cn])

✨ You're running DeepEval's latest Tone Attunement [GEval] Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Cultural Neutrality [GEval] Metric! (using gpt-4.1-students (Azure), 
strict=False, async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Tone Attunement [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The Input expresses intense fear and urgency, requiring a highly empathetic and supportive response. The Actual Output demonstrates strong empathy, acknowledges the user's distress, and maintains a consistent, supportive tone throughout. It offers clear, actionable advice with appropriate formality and sensitivity, aligning well with the user's emotional state and needs., error: None)
  - ✅ Cultural Neutrality [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The output demonstrates strong alignment with the evaluation steps by providing neutral, inclusive, and supportive advice without making assumptions about the user's background, legal status, literacy, gender, or cultural context. It avoids bias and alienating statements, focusing solely on safety and practical steps. The response is e

⚠ WARNING: No hyperparameters logged.
» ]8;id=329943;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 15.91s | token cost: None)
» Test Results (21 total tests):
   » Pass Rate: 95.24% | Passed: 20 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [198]:
results = []
for test_result in humanitarian_results.test_results:
    row = {
        "input": test_result.input,
        "bot_output": test_result.actual_output,
    }
    for metric_data in test_result.metrics_data:
        row[metric_data.name] = metric_data.score
        row[metric_data.name + "_reason"] = metric_data.reason
    results.append(row)
humanitarian_results_df = pd.DataFrame(results)

In [199]:
humanitarian_results_df.head()

,input,bot_output,Tone Attunement [GEval],Tone Attunement [GEval]_reason,Cultural Neutrality [GEval],Cultural Neutrality [GEval]_reason
0,"I am running away from my abuser, who is searching for me and my children. I do not know where I am and I am very afraid. What can I do?","I'm really sorry to hear that you're in this situation. Your safety and the safety of your children are the most important things right now. Here are some steps you can consider:\n\n1. Contact Emergency Services: If you are in immediate danger, please call emergency services (such as 911) right away.\n\n2. Reach Out to a Domestic Violence Hotline: They can provide you with confidential support, advice, and resources tailored to your situation.\n\n3. Find a Safe Place: If possible, go to a shelter or a trusted friend's or family member's home where your abuser cannot find you.\n\n4. Keep Important Documents and Essentials: If you can, keep identification, important documents, some money, and essentials for you and your children with you.\n\n5. Consider Legal Protection: You might want to look into obtaining a restraining order or protection order against your abuser.\n\n6. Use Technology Safely: Be cautious with your phone and online activity, as your abuser might track you through these means.\n\nIf you want, I can help you find specific resources or information within the HIA app that might assist you. Please let me know how I can support you.",1.0,"The Input expresses intense fear and urgency, requiring a highly empathetic and supportive response. The Actual Output demonstrates strong empathy, acknowledges the user's distress, and maintains a consistent, supportive tone throughout. It offers clear, actionable advice with appropriate formality and sensitivity, aligning well with the user's emotional state and needs.",1.0,"The output demonstrates strong alignment with the evaluation steps by providing neutral, inclusive, and supportive advice without making assumptions about the user's background, legal status, literacy, gender, or cultural context. It avoids bias and alienating statements, focusing solely on safety and practical steps. The response is empathetic and does not reference any specific demographic details from the input."
1,"I have been to 3 hospitals, but they refuse me because I am undocumented. I am in immense pain and need immediate attention. I am in Amsterdam. What do I do?","Hello, I am Aidly, the HIA chatbot.\n\nIf you are undocumented in Amsterdam and in immense pain needing immediate medical attention, you still have the right to necessary medical care in the Netherlands, even without residency status.\n\nHere are some steps you can take:\n\n- Visit the GGD Center for Sexual Health in Amsterdam if your issue relates to sexual health; they provide free care without needing ID or health insurance. You can make an appointment by phone at (020) 555 5822 or via their website: https://afspraak.ggdaphrodite.nl/?lang=en\n\n- For urgent medical issues outside regular hours, you can call or visit the Huisartsenpost Amsterdam at +31 88 - 00 30 600 (open 17:00 - 08:00). If it is life-threatening, call 112 immediately.\n\n- You can also reach out to the Netherlands Red Cross Humanitarian Service Point in Amsterdam, where they examine your situation and seek possible solutions regardless of your papers.\n\n- If you are unable to pay for medically necessary care, healthcare providers can appeal to subsidy schemes for uninsured persons.\n\n- If you feel your pain is an emergency, do not hesitate to call 112 or go to the Emergency Room of the nearest hospital.\n\nPlease seek help immediately through these channels. Your health and safety are important.",0.9,"The chatbot's response maintains a formal yet compassionate tone, which is appropriate given the user's distress and urgent medical need. It provides clear, actionable steps and reassures the user of their right to care, demonstrating empathy. The tone is consistent throughout, though a slightl

In [201]:
# Import the processed survey data
human_data = pd.read_csv("../data/human_annotation_scores.csv")
print(human_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Question  21 non-null     object 
 1   Answer    21 non-null     object 
 2   TA_score  21 non-null     float64
 3   TA_label  21 non-null     bool   
 4   CN_score  21 non-null     float64
 5   CN_label  21 non-null     bool   
dtypes: bool(2), float64(2), object(2)
memory usage: 846.0+ bytes


None

## Tone Attunement

In [203]:
total_entries = len(human_data)
print(f"Total number of entries: {total_entries}")

# Pass rates from human labels
ta_pass_rate = (human_data["TA_label"] == True).mean()
print(f"\nHuman TA pass rate: {ta_pass_rate:.2%}")

# Pass rates from DeepEval at default 0.5
ta_de_pass_rate = (humanitarian_results_df["Tone Attunement [GEval]"] >= 0.5).mean()
print(f"\nDeepEval TA pass rate @ 0.5: {ta_de_pass_rate:.2%}")

# Score distributions
print(f"\nTA scores (sorted ascending):")
print(sorted(humanitarian_results_df["Tone Attunement [GEval]"].tolist()))

Total number of entries: 21

Human TA pass rate: 52.38%

DeepEval TA pass rate @ 0.5: 95.24%

TA scores (sorted ascending):

[0.2, 0.5, 0.6, 0.7, 0.7, 0.7, 0.7, 0.8, 0.8, 0.8, 0.8, 0.8, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 1.0, 1.0]

### Do we agree on which instances pass and which fail?

Passing labels

In [208]:
print(f"Passing rows TA (human annotators): {human_data[human_data['TA_label'] == True].index.tolist()}")
print(f"Passing rows TA (DeepEval @ 0.5): {humanitarian_results_df[humanitarian_results_df['Tone Attunement [GEval]'] >= 0.5].index.tolist()}")
print(f"Rows which don't match (TA): {set(human_data[human_data['TA_label'] == True].index) ^ set(humanitarian_results_df[humanitarian_results_df['Tone Attunement [GEval]'] >= 0.5].index)}")

Passing rows TA (human annotators): [0, 1, 4, 5, 9, 11, 14, 16, 17, 19, 20]

Passing rows TA (DeepEval @ 0.5): [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Rows which don't match (TA): {2, 3, 5, 6, 7, 8, 10, 12, 13, 15, 18}

Since there are a lot of mismatches, instead of inspecting them one by one, we will look for the most pronounced cases - i.e., cases where humans confidently fail and the metric confidently passed.

In [225]:
# Rename so column matches
human_data_renamed = human_data.rename(columns={"Question": "input"})

# Merge
merged_ta = humanitarian_results_df.merge(
    human_data_renamed,
    on="input",
    how="inner"
)

# Mismatches where human said fail (label=False) but metric passed (score >= 0.5)
mismatch_indices = [2, 3, 5, 6, 7, 8, 10, 12, 13, 15, 18]

gap_data = []
for idx in mismatch_indices:
    human = merged_ta.loc[idx, "TA_score"]
    metric = merged_ta.loc[idx, "Tone Attunement [GEval]"]
    gap_data.append({
        "row": idx,
        "human": human,
        "metric": metric,
        "gap": metric - human
    })

gap_df = pd.DataFrame(gap_data).sort_values("gap", ascending=False)
print(gap_df.to_string(index=False))

row   human  metric      gap
  13 0.39375     0.7  0.30625
   8 0.51750     0.8  0.28250
  15 0.64375     0.9  0.25625
   6 0.71375     0.9  0.18625
  10 0.51750     0.7  0.18250
  12 0.62500     0.8  0.17500
   3 0.37500     0.5  0.12500
  18 0.78500     0.9  0.11500
   7 0.83875     0.8 -0.03875
   2 0.76875     0.7 -0.06875
   5 0.41000     0.2 -0.21000

In [226]:
for idx in [13, 8, 15]:
    print(f"Row {idx}:")
    print(merged_ta.loc[idx, ["input"]])
    print(merged_ta.loc[idx, ["bot_output"]])
    print(merged_ta.loc[idx, ["Tone Attunement [GEval]_reason"]])

Row 13:

input    You already told me about the day shelters. I know about those. I need an actual place to sleep tonight 
with my children. Stop giving me the same information!
Name: 13, dtype: object

bot_output    I understand you urgently need a place to sleep tonight with your children. For immediate shelter, 
you should go to the municipality ("gemeente") where you are or will be staying. Municipalities are responsible for
arranging shelter for displaced persons from Ukraine, including families with children. \n\nIf you are with 
unaccompanied children, Nidos arranges shelter in host families, preferably with the same cultural background. For 
families, the municipality will help find suitable accommodation.\n\nIf you are in immediate danger or need urgent 
help, please call emergency services at 112.\n\nFor more information and to find the nearest registration point, 
you can visit the Red Cross website with updated shelter information for Ukrainian refugees:  \n- 
[English](https://www.rodekruis.nl/english-information/)  \n- [Ukrainian](https://www.rodekruis.nl/ukrainian/)  \n-
[Russian](https://www.rodekruis.nl/russian/)\n\nPlease go to the municipality as soon as possible to get help with 
shelter tonight. If you tell me your location, I can try to help you find the nearest registration point.
Name: 13, dtype: object

Tone Attunement [GEval]_reason    The response acknowledges the user's urgent need and frustration, showing some 
empathy and adjusting the information to focus on immediate shelter options. The tone is formal and consistent, 
which matches the seriousness of the user's situation, but it could be warmer and more responsive to the user's 
emotional distress. The chatbot repeats some general information and could have more directly addressed the user's 
frustration about repeated advice. Overall, the response is helpful and mostly aligned, but lacks a stronger 
empathetic tone and a more conversational style to fully match the user's emotional state.
Name: 13, dtype: object

Row 8:

...

Tone Attunement [GEval]_reason    The response acknowledges the user's distress and provides practical, relevant 
steps for immediate help, demonstrating empathy and a supportive tone. The formality is appropriate for the 
seriousness of the situation, and the tone remains consistent throughout. However, the emotional tone could be more
closely matched to the user's high level of distress; the empathy, while present, is somewhat limited to the 
opening sentence and could be deepened to better align with the user's emotional state.
Name: 8, dtype: object

Row 15:

input    I am from Ukraine and I work in a factory. I live with my other colleagues, and my boss pays for my rent, 
utilities, and gives me an allowance. I want to go home this Christmas to visit my family, but he refuses because 
he says it's the busiest season and if I leave everyone will want to go home as well, so we will all lose our jobs.
Can you help somehow?
Name: 15, dtype: object

bot_output    Hello! I am Aidly, the HIA chatbot.\n\nFrom the information available, as a displaced person from 
Ukraine working in the Netherlands, you have the same labour rights as Dutch employees, including the right to a 
safe working space and fair treatment. Your employer should not force you to work against your will or prevent you 
from taking reasonable time off, such as for family visits during holidays.\n\nIf your employer is refusing to 
allow you to visit your family for Christmas and you feel pressured or unsafe, you can seek help. There are 
organizations like FairWork that specialize in helping people with work problems or labour exploitation. You can 
contact FairWork at +31 20 760 08 09 or visit https://www.fairwork.nu/en/homepage/ for assistance.\n\nAlso, you can
share with your employer the flyer "Know your rights when you work," available in English and Ukrainian, to inform 
them about your rights: 
https://www.comensha.nl/media/jizlbk4d/2026-know-your-rights-flyer-oek-ontheemden-eng.pdf\n\nIf the situation is 
urgent or you feel forced to work against your will, you can ask for immediate help through FairWork.\n\nRemember, 
you should also report your departure if you live in a shelter, but since you live with colleagues and your boss 
pays rent, make sure to inform the relevant reception or municipality if applicable.\n\nIf you want, I can help you
find more detailed advice or contacts. Would you like that?
Name: 15, dtype: object

Tone Attunement [GEval]_reason    The user's input is emotionally vulnerable, expressing concern and a desire for 
help in a difficult work situation. The Actual Output responds with a formal yet empathetic tone, providing clear 
information about rights, resources, and next steps. The tone is consistent throughout and appropriately balances 
professionalism with supportive language, such as offering further help and acknowledging the user's situation. A 
slightly warmer or more explicitly empathetic opening could improve alignment, but overall the response is 
well-matched to the user's emotional state.
Name: 15, dtype: object

A case could be made here with the chatbot not being more empathetic, but I agree with the LLM scoring and reasoning. Humans seem to be expecting more emotional engagement than the metric (and arguably than is appropriate for an informational chatbot). This can be considered a limitation, and will be appropriately discussed in the paper. Krippendorff's alpha to be computed in a separate stage; gap between metric and human strictness expected to surface there.

### Applying the percentile method with target

In [235]:
count = (human_data["TA_label"] == True).sum()
ta_percentile = (count / total_entries) * 100
sorted_scores = sorted(humanitarian_results_df["Tone Attunement [GEval]"].tolist())  # ascending
index = int(len(sorted_scores) * (1 - ta_percentile / 100))
print(f"Tone Attunement threshold for {ta_percentile:.2f}% percentile: {sorted_scores[index]:.2f}")

Tone Attunement threshold for 52.38% percentile: 0.80

## Cultural Neutrality

In [210]:
total_entries = len(human_data)
print(f"Total number of entries: {total_entries}")

# Pass rates from human labels
cn_pass_rate = (human_data["CN_label"] == True).mean()
print(f"\nHuman CN pass rate: {cn_pass_rate:.2%}")

# Pass rates from DeepEval at default 0.5
cn_de_pass_rate = (humanitarian_results_df["Cultural Neutrality [GEval]"] >= 0.5).mean()
print(f"\nDeepEval CN pass rate @ 0.5: {cn_de_pass_rate:.2%}")

# Score distributions
print(f"\nCN scores (sorted ascending):")
print(sorted(humanitarian_results_df["Cultural Neutrality [GEval]"].tolist()))

Total number of entries: 21

Human CN pass rate: 90.48%

DeepEval CN pass rate @ 0.5: 100.00%

CN scores (sorted ascending):

[0.6, 0.7, 0.9, 0.9, 0.9, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

### Do we agree on which instances pass and which fail?

In [231]:
print(f"Passing rows CN (human annotators): {human_data[human_data['CN_label'] == True].index.tolist()}")
print(f"Passing rows CN (DeepEval @ 0.5): {humanitarian_results_df[humanitarian_results_df['Cultural Neutrality [GEval]'] >= 0.5].index.tolist()}")
print(f"Rows which don't match (CN): {set(human_data[human_data['CN_label'] == True].index) ^ set(humanitarian_results_df[humanitarian_results_df['Cultural Neutrality [GEval]'] >= 0.5].index)}")

Passing rows CN (human annotators): [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20]

Passing rows CN (DeepEval @ 0.5): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Rows which don't match (CN): {8, 18}

In [232]:
# Rename so column matches
human_data_renamed = human_data.rename(columns={"Question": "input"})

# Merge
merged_cn = humanitarian_results_df.merge(
    human_data_renamed,
    on="input",
    how="inner"
)

# Mismatches where human said fail (label=False) but metric passed (score >= 0.5)
mismatch_indices = [8,18]

gap_data = []
for idx in mismatch_indices:
    human = merged_cn.loc[idx, "CN_score"]
    metric = merged_cn.loc[idx, "Cultural Neutrality [GEval]"]
    gap_data.append({
        "row": idx,
        "human": human,
        "metric": metric,
        "gap": metric - human
    })

gap_df = pd.DataFrame(gap_data).sort_values("gap", ascending=False)
print(gap_df.to_string(index=False))

row   human  metric     gap
   8 0.76400     1.0 0.23600
  18 0.94575     1.0 0.05425

The gap value is insignificant, and therefore no further analysis will be performed.

### Applying the percentile method with target

In [234]:
count = (human_data["CN_label"] == True).sum()
cn_percentile = (count / total_entries) * 100
sorted_scores = sorted(humanitarian_results_df["Cultural Neutrality [GEval]"].tolist())  # ascending
index = int(len(sorted_scores) * (1 - cn_percentile / 100))
print(f"Cultural Neutrality threshold for {cn_percentile:.2f}% percentile: {sorted_scores[index]:.2f}")

Cultural Neutrality threshold for 90.48% percentile: 0.90